# Experiment 9: Lid-Driven Cavity Flow

**Aim:** Solve the incompressible cavity problem using the vorticity-streamfunction formulation and analyse the primary recirculation.

**Numerical method:** Explicit vorticity transport, SOR solution of the streamfunction Poisson equation, and wall-vorticity boundary conditions.

> **Scope:** Grid-independence and comparison with benchmark centreline data remain required student validation tasks.

This Google Colab notebook is self-contained. It reconstructs the tested solver inside the temporary Colab runtime, runs it, displays quantitative plots and an inline field contour, and packages every output file for download.

> **Colab workflow:** Choose a run mode, use **Runtime > Run all**, then download the results ZIP from the final cell. `quick` is only a workflow check; use `lab` or an approved `custom` case for submission.

In [1]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "07_Shallow_Water_Wave# Experiment 8 — Rayleigh–Taylor Instability with a Volume-of-Fluid Finite-Volume Solver\n",
    "\n",
    "\n",
    "It simulates the **Rayleigh–Taylor instability (RTI)** of a heavy fluid resting on a light fluid in a\n",
    "2-D box, using\n",
    "\n",
    "* a **volume-of-fluid (VOF)** colour function advected with a first-order **upwind** finite-volume scheme,\n",
    "* a **staggered (MAC) grid** incompressible Navier–Stokes solver,\n",
    "* a **Chorin projection** with a sparse-direct Poisson solve,\n",
    "* the **Boussinesq approximation** for the density difference.\n"
   ]
  },
  {
   "cell_type": "code",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-27T17:48:58.746913Z",
     "iopub.status.busy": "2026-10-27T17:48:58.746913Z",
     "iopub.status.idle": "2026-10-27T17:48:59.281221Z",
     "shell.execute_reply": "2026-10-27T17:48:59.281221Z"
    },
    "ExecuteTime": {
     "end_time": "2026-10-27T17:49:58.246530Z",
     "start_time": "2026-10-27T17:49:57.653098Z"
    }
   },
   "source": [
    "import time\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "from scipy.sparse import diags, eye, kron\n",
    "from scipy.sparse.linalg import factorized\n",
    "from scipy.optimize import minimize_scalar\n",
    "\n",
    "nx, ny = 48, 96\n",
    "Lx, Ly = 1.0, 2.0\n",
    "dx, dy = Lx / nx, Ly / ny\n",
    "rho_light, rho_heavy, g, nu = 1.0, 1.2, 9.81, 0.005\n",
    "rho_ref = 0.5 * (rho_light + rho_heavy)\n",
    "t_end = 2.1\n",
    "amp0 = 0.04                      # initial interface amplitude (m)\n",
    "\n",
    "A_atwood = (rho_heavy - rho_light) / (rho_heavy + rho_light)\n",
    "k_wave = 2 * np.pi / Lx\n",
    "sigma_inviscid = np.sqrt(A_atwood * g * k_wave)\n",
    "sigma_viscous = np.sqrt(A_atwood * g * k_wave + (nu * k_wave**2) ** 2) - nu * k_wave**2\n",
    "\n",
    "print(f\"Grid            : {nx} x {ny}, dx = {dx:.5f} m, dy = {dy:.5f} m\")\n",
    "print(f\"Atwood number   : A = {A_atwood:.5f}\")\n",
    "print(f\"Wavenumber      : k = {k_wave:.5f} 1/m (lambda = {Lx} m)\")\n",
    "print(f\"sigma (inviscid): {sigma_inviscid:.4f} 1/s   -> e-folding time {1/sigma_inviscid:.3f} s\")\n",
    "print(f\"sigma (viscous approx.): {sigma_viscous:.4f} 1/s\")\n",
    "print(f\"Viscous damping rate nu*k^2 = {nu*k_wave**2:.4f} 1/s\")"
   ],
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Grid            : 48 x 96, dx = 0.02083 m, dy = 0.02083 m\n",
      "Atwood number   : A = 0.09091\n",
      "Wavenumber      : k = 6.28319 1/m (lambda = 1.0 m)\n",
      "sigma (inviscid): 2.3672 1/s   -> e-folding time 0.422 s\n",
      "sigma (viscous approx.): 2.1780 1/s\n",
      "Viscous damping rate nu*k^2 = 0.1974 1/s\n"
     ]
    }
   ],
   "execution_count": 1
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### Initial condition and pressure operator\n",
    "The discrete Laplacian is assembled with Kronecker products: periodic in $x$ (corner entries), Neumann in $y$ (`-1` on the first/last diagonal entries). The first row is replaced by an identity row to pin the pressure level and remove the null space; the matrix is factorised once."
   ]
  },
  {
   "cell_type": "code",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-27T17:48:59.297054Z",
     "iopub.status.busy": "2026-10-27T17:48:59.297054Z",
     "iopub.status.idle": "2026-10-27T17:48:59.328999Z",
     "shell.execute_reply": "2026-10-27T17:48:59.328999Z"
    },
    "ExecuteTime": {
     "end_time": "2026-10-27T17:49:58.270332Z",
     "start_time": "2026-10-27T17:49:58.254548Z"
    }
   },
   "source": [
    "def initial_condition(nx, ny, Lx, Ly, amp0=0.04):\n",
    "    dx, dy = Lx / nx, Ly / ny\n",
    "    x = (np.arange(nx) + 0.5) * dx\n",
    "    y = (np.arange(ny) + 0.5) * dy\n",
    "    interface = 1.0 + amp0 * np.cos(2 * np.pi * x / Lx)\n",
    "    alpha = np.clip((y[:, None] + dy / 2 - interface[None, :]) / dy, 0, 1)\n",
    "    u = np.zeros((ny, nx))\n",
    "    v = np.zeros((ny + 1, nx))\n",
    "    return x, y, alpha, u, v\n",
    "\n",
    "\n",
    "def build_pressure_solver(nx, ny, dx, dy):\n",
    "    Lx_op = diags([np.ones(nx - 1), -2 * np.ones(nx), np.ones(nx - 1)], [-1, 0, 1]).tolil()\n",
    "    Lx_op[0, -1] = Lx_op[-1, 0] = 1\n",
    "    Ly_op = diags([np.ones(ny - 1), -2 * np.ones(ny), np.ones(ny - 1)], [-1, 0, 1]).tolil()\n",
    "    Ly_op[0, 0] = Ly_op[-1, -1] = -1\n",
    "    lap = kron(eye(ny), Lx_op / dx**2) + kron(Ly_op / dy**2, eye(nx))\n",
    "    lap = lap.tolil()\n",
    "    lap[0, :] = 0\n",
    "    lap[0, 0] = 1\n",
    "    return factorized(lap.tocsc())"
   ],
   "outputs": [],
   "execution_count": 2
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "VOF advection (first-order upwind)\n",
    "Face fluxes $F=u\\,\\alpha_{upwind}$ and $G=v\\,\\alpha_{upwind}$; at the top/bottom walls the ghost value equals the adjacent cell (and $v=0$ there, so no flux crosses)."
   ]
  },
  {
   "cell_type": "code",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-27T17:48:59.332247Z",
     "iopub.status.busy": "2026-10-27T17:48:59.332247Z",
     "iopub.status.idle": "2026-10-27T17:48:59.344663Z",
     "shell.execute_reply": "2026-10-27T17:48:59.344663Z"
    },
    "ExecuteTime": {
     "end_time": "2026-10-27T17:49:58.290559Z",
     "start_time": "2026-10-27T17:49:58.274892Z"
    }
   },
   "source": [
    "def advect_alpha(alpha, u, v, dt, dx, dy):\n",
    "    fx = u * np.where(u >= 0, np.roll(alpha, 1, axis=1), alpha)\n",
    "    lower = np.vstack((alpha[:1], alpha))\n",
    "    upper = np.vstack((alpha, alpha[-1:]))\n",
    "    fy = v * np.where(v >= 0, lower, upper)\n",
    "    return alpha - dt * ((np.roll(fx, -1, axis=1) - fx) / dx\n",
    "                         + (fy[1:] - fy[:-1]) / dy)"
   ],
   "outputs": [],
   "execution_count": 3
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### Momentum predictors\n",
    "$u^*$ and $v^*$ include upwind convection, viscous diffusion and (for $v$) Boussinesq buoyancy. Pressure is not included yet."
   ]
  },
  {
   "cell_type": "code",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-27T17:48:59.344663Z",
     "iopub.status.busy": "2026-10-27T17:48:59.344663Z",
     "iopub.status.idle": "2026-10-27T17:48:59.360409Z",
     "shell.execute_reply": "2026-10-27T17:48:59.360409Z"
    },
    "ExecuteTime": {
     "end_time": "2026-10-27T17:49:58.310788Z",
     "start_time": "2026-10-27T17:49:58.295069Z"
    }
   },
   "source": [
    "def predict_u(u, v, dt, dx, dy, nu):\n",
    "    ue = 0.5 * (u + np.roll(u, -1, axis=1))\n",
    "    uw = 0.5 * (u + np.roll(u, 1, axis=1))\n",
    "    un = np.roll(u, -1, axis=0)\n",
    "    us = np.roll(u, 1, axis=0)\n",
    "    vn = 0.5 * (v[1:] + np.roll(v[1:], 1, axis=1))\n",
    "    vs = 0.5 * (v[:-1] + np.roll(v[:-1], 1, axis=1))\n",
    "    un[-1] = u[-1]\n",
    "    us[0] = u[0]\n",
    "    flux_ue = ue * np.where(ue >= 0, u, np.roll(u, -1, axis=1))\n",
    "    flux_uw = uw * np.where(uw >= 0, np.roll(u, 1, axis=1), u)\n",
    "    flux_un = vn * np.where(vn >= 0, u, un)\n",
    "    flux_us = vs * np.where(vs >= 0, us, u)\n",
    "    conv_u = (flux_ue - flux_uw) / dx + (flux_un - flux_us) / dy\n",
    "    lap_u = (np.roll(u, -1, axis=1) - 2*u + np.roll(u, 1, axis=1)) / dx**2\n",
    "    lap_u += (un - 2*u + us) / dy**2\n",
    "    return u + dt * (-conv_u + nu * lap_u)\n",
    "\n",
    "\n",
    "def predict_v(u, v, alpha, dt, dx, dy, nu, g, rho_light, rho_heavy, rho_ref):\n",
    "    vm = v[1:-1]\n",
    "    ve = 0.5 * (np.roll(u[:-1], -1, axis=1) + np.roll(u[1:], -1, axis=1))\n",
    "    vw = 0.5 * (u[:-1] + u[1:])\n",
    "    vne = 0.5 * (vm + v[2:])\n",
    "    vsw = 0.5 * (vm + v[:-2])\n",
    "    flux_ve = ve * np.where(ve >= 0, vm, np.roll(vm, -1, axis=1))\n",
    "    flux_vw = vw * np.where(vw >= 0, np.roll(vm, 1, axis=1), vm)\n",
    "    flux_vn = vne * np.where(vne >= 0, vm, v[2:])\n",
    "    flux_vs = vsw * np.where(vsw >= 0, v[:-2], vm)\n",
    "    conv_v = (flux_ve - flux_vw) / dx + (flux_vn - flux_vs) / dy\n",
    "    lap_v = (np.roll(vm, -1, axis=1) - 2*vm + np.roll(vm, 1, axis=1)) / dx**2\n",
    "    lap_v += (v[2:] - 2*vm + v[:-2]) / dy**2\n",
    "    buoyancy = -g * (rho_heavy - rho_light) * (alpha[:-1] + alpha[1:] - 1) / (2*rho_ref)\n",
    "    v_star = v.copy()\n",
    "    v_star[1:-1] = vm + dt * (-conv_v + nu * lap_v + buoyancy)\n",
    "    return v_star"
   ],
   "outputs": [],
   "execution_count": 4
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### Pressure projection and divergence\n",
    "The Poisson equation enforces $\\nabla\\cdot\\mathbf u^{n+1}=0$ to solver precision; the divergence function is also used as a diagnostic."
   ]
  },
  {
   "cell_type": "code",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-27T17:48:59.360409Z",
     "iopub.status.busy": "2026-10-27T17:48:59.360409Z",
     "iopub.status.idle": "2026-10-27T17:48:59.376191Z",
     "shell.execute_reply": "2026-10-27T17:48:59.376191Z"
    },
    "ExecuteTime": {
     "end_time": "2026-10-27T17:49:58.330968Z",
     "start_time": "2026-10-27T17:49:58.315011Z"
    }
   },
   "source": [
    "def divergence_field(u, v, dx, dy):\n",
    "    d = (np.roll(u, -1, axis=1) - u) / dx\n",
    "    d += (v[1:] - v[:-1]) / dy\n",
    "    return d\n",
    "\n",
    "\n",
    "def project(u_star, v_star, dt, dx, dy, rho_ref, solve_pressure):\n",
    "    ny, nx = u_star.shape\n",
    "    divergence = divergence_field(u_star, v_star, dx, dy)\n",
    "    rhs = (rho_ref / dt * divergence).ravel()\n",
    "    rhs[0] = 0\n",
    "    p = solve_pressure(rhs).reshape(ny, nx)\n",
    "    u = u_star - (dt / (rho_ref * dx)) * (p - np.roll(p, 1, axis=1))\n",
    "    v = v_star\n",
    "    v[1:-1] -= (dt / (rho_ref * dy)) * (p[1:] - p[:-1])\n",
    "    return u, v, p"
   ],
   "outputs": [],
   "execution_count": 5
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "* **Spike / bubble penetration:** the interface is the $\\alpha=0.5$ level. The lowest crossing over all columns gives the spike tip $y_s$, the highest crossing gives the bubble tip $y_b$. Penetrations are $h_s=1-y_s$, $h_b=y_b-1$ and the mean amplitude is $a=\\tfrac12(h_s+h_b)$ (initially $a\\approx0.04$ m).\n",
    "* **Kinetic energy** $E_k\\approx\\tfrac12\\rho_{ref}\\,(\\sum u^2+\\sum v^2)\\Delta x\\Delta y$ (per unit depth, J/m).\n",
    "* **Interface thickness measure:** area of cells with $0.05<\\alpha<0.95$ (numerical smearing).\n",
    "* **Mass:** $\\sum\\alpha\\,\\Delta x\\Delta y$ (heavy-fluid area)."
   ]
  },
  {
   "cell_type": "code",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-27T17:48:59.376191Z",
     "iopub.status.busy": "2026-10-27T17:48:59.376191Z",
     "iopub.status.idle": "2026-10-27T17:48:59.391861Z",
     "shell.execute_reply": "2026-10-27T17:48:59.391861Z"
    },
    "ExecuteTime": {
     "end_time": "2026-10-27T17:49:58.350974Z",
     "start_time": "2026-10-27T17:49:58.335357Z"
    }
   },
   "source": [
    "def interface_tips(alpha, y, dy):\n",
    "    # lowest alpha=0.5 crossing (spike tip) and highest crossing (bubble tip), linearly interpolated\n",
    "    ny, nx = alpha.shape\n",
    "    cols = np.arange(nx)\n",
    "    j = np.argmax(alpha >= 0.5, axis=0)                 # first heavy cell from the bottom\n",
    "    jm = np.maximum(j - 1, 0)\n",
    "    a0, a1 = alpha[jm, cols], alpha[j, cols]\n",
    "    den = np.where(np.abs(a1 - a0) < 1e-14, 1.0, a1 - a0)\n",
    "    yc_low = np.where(j == 0, y[0], y[jm] + (0.5 - a0) / den * dy)\n",
    "    jl = ny - 1 - np.argmax((alpha < 0.5)[::-1], axis=0)  # last light cell from the bottom\n",
    "    jp = np.minimum(jl + 1, ny - 1)\n",
    "    b0, b1 = alpha[jl, cols], alpha[jp, cols]\n",
    "    den = np.where(np.abs(b1 - b0) < 1e-14, 1.0, b1 - b0)\n",
    "    yc_high = np.where(jl == ny - 1, y[-1], y[jl] + (0.5 - b0) / den * dy)\n",
    "    return yc_low.min(), yc_high.max()\n",
    "\n",
    "\n",
    "def diagnostics(alpha, u, v, y, dx, dy, rho_ref, div_max_now):\n",
    "    y_spike, y_bubble = interface_tips(alpha, y, dy)\n",
    "    hs, hb = 1.0 - y_spike, y_bubble - 1.0\n",
    "    return dict(h_spike=hs, h_bubble=hb, amp=0.5 * (hs + hb),\n",
    "                max_speed=max(np.max(np.abs(u)), np.max(np.abs(v))),\n",
    "                ke=0.5 * rho_ref * (np.sum(u**2) + np.sum(v**2)) * dx * dy,\n",
    "                mass=alpha.sum() * dx * dy,\n",
    "                smear=np.sum((alpha > 0.05) & (alpha < 0.95)) * dx * dy,\n",
    "                div=div_max_now)"
   ],
   "outputs": [],
   "execution_count": 6
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### Time-stepping driver\n",
    "`run_rti` reproduces the original `while` loop exactly: advect $\\alpha$ with the old velocity, predict $u^*,v^*$ (buoyancy uses the *new* $\\alpha$), project, check bounds on $\\alpha$, advance time, store snapshots when $t\\ge m\\,t_{end}/4$."
   ]
  },
  {
   "cell_type": "code",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-27T17:48:59.391861Z",
     "iopub.status.busy": "2026-10-27T17:48:59.391861Z",
     "iopub.status.idle": "2026-10-27T17:48:59.407960Z",
     "shell.execute_reply": "2026-10-27T17:48:59.407960Z"
    },
    "ExecuteTime": {
     "end_time": "2026-10-27T17:49:58.371266Z",
     "start_time": "2026-10-27T17:49:58.355171Z"
    }
   },
   "source": [
    "def run_rti(nx, ny, Lx=1.0, Ly=2.0, t_end=2.1, nu=0.005, g=9.81,\n",
    "            rho_light=1.0, rho_heavy=1.2, amp0=0.04, verbose=False):\n",
    "    dx, dy = Lx / nx, Ly / ny\n",
    "    rho_ref = 0.5 * (rho_light + rho_heavy)\n",
    "    x, y, alpha, u, v = initial_condition(nx, ny, Lx, Ly, amp0)\n",
    "    solve_pressure = build_pressure_solver(nx, ny, dx, dy)\n",
    "\n",
    "    snapshots = [(0.0, alpha.copy())]\n",
    "    initial_mass = alpha.sum() * dx * dy\n",
    "    hist = {k: [] for k in [\"t\", \"h_spike\", \"h_bubble\", \"amp\", \"max_speed\", \"ke\", \"mass\", \"smear\", \"div\"]}\n",
    "    d0 = diagnostics(alpha, u, v, y, dx, dy, rho_ref, 0.0)\n",
    "    hist[\"t\"].append(0.0)\n",
    "    for k, val in d0.items():\n",
    "        hist[k].append(val)\n",
    "    snap_diag = [dict(t=0.0, **d0)]\n",
    "    t, steps, maximum_divergence = 0.0, 0, 0.0\n",
    "    t_wall = time.time()\n",
    "    while t < t_end - 1e-12:\n",
    "        adv_rate = np.max(np.abs(u)) / dx + np.max(np.abs(v)) / dy\n",
    "        dt = min(0.003, 0.3 / max(adv_rate, 1e-12),\n",
    "                 0.45 / (2 * nu * (1/dx**2 + 1/dy**2)), t_end - t)\n",
    "\n",
    "        alpha = advect_alpha(alpha, u, v, dt, dx, dy)\n",
    "        u_star = predict_u(u, v, dt, dx, dy, nu)\n",
    "        v_star = predict_v(u, v, alpha, dt, dx, dy, nu, g, rho_light, rho_heavy, rho_ref)\n",
    "        u, v, p = project(u_star, v_star, dt, dx, dy, rho_ref, solve_pressure)\n",
    "\n",
    "        current_div = divergence_field(u, v, dx, dy)\n",
    "        maximum_divergence = max(maximum_divergence, np.max(np.abs(current_div)))\n",
    "        if alpha.min() < -1e-9 or alpha.max() > 1 + 1e-9:\n",
    "            raise RuntimeError(\"Volume fraction left physical bounds\")\n",
    "        t += dt\n",
    "        steps += 1\n",
    "        d = diagnostics(alpha, u, v, y, dx, dy, rho_ref, np.max(np.abs(current_div)))\n",
    "        hist[\"t\"].append(t)\n",
    "        for k, val in d.items():\n",
    "            hist[k].append(val)\n",
    "        if t >= len(snapshots) * (t_end / 4) - 1e-12:\n",
    "            snapshots.append((t, alpha.copy()))\n",
    "            snap_diag.append(dict(t=t, **d))\n",
    "    wall = time.time() - t_wall\n",
    "    mass = alpha.sum() * dx * dy\n",
    "    return dict(nx=nx, ny=ny, dx=dx, dy=dy, x=x, y=y, alpha=alpha, u=u, v=v, p=p,\n",
    "                snapshots=snapshots, snap_diag=snap_diag, hist={k: np.array(val) for k, val in hist.items()},\n",
    "                steps=steps, t=t, initial_mass=initial_mass, mass=mass,\n",
    "                maximum_divergence=maximum_divergence, wall=wall)"
   ],
   "outputs": [],
   "execution_count": 7
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Main simulation (48 × 96 grid)\n",
    "\n",
    "This cell runs the case defined in the original script and prints the same summary lines."
   ]
  },
  {
   "cell_type": "code",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-27T17:48:59.407960Z",
     "iopub.status.busy": "2026-10-27T17:48:59.407960Z",
     "iopub.status.idle": "2026-10-27T17:49:00.089134Z",
     "shell.execute_reply": "2026-10-27T17:49:00.089134Z"
    },
    "ExecuteTime": {
     "end_time": "2026-10-27T17:49:59.116279Z",
     "start_time": "2026-10-27T17:49:58.377196Z"
    }
   },
   "source": [
    "res = run_rti(nx, ny, Lx, Ly, t_end, nu, g, rho_light, rho_heavy, amp0)\n",
    "x, y = res[\"x\"], res[\"y\"]\n",
    "alpha, u, v = res[\"alpha\"], res[\"u\"], res[\"v\"]\n",
    "snapshots, hist = res[\"snapshots\"], res[\"hist\"]\n",
    "\n",
    "print(f\"Grid={nx}x{ny}; steps={res['steps']}; time={res['t']:.4f}\")\n",
    "print(f\"Initial heavy-fluid area={res['initial_mass']:.12f}\")\n",
    "print(f\"Final heavy-fluid area={res['mass']:.12f}\")\n",
    "print(f\"Relative mass error={abs(res['mass'] - res['initial_mass'])/res['initial_mass']:.3e}\")\n",
    "print(f\"Volume fraction range=[{alpha.min():.6f}, {alpha.max():.6f}]\")\n",
    "print(f\"Maximum projected divergence={res['maximum_divergence']:.3e}\")\n",
    "print(f\"Max speed={max(np.max(np.abs(u)), np.max(np.abs(v))):.6f}\")\n",
    "print(f\"(wall-clock time of this run: {res['wall']:.1f} s)\")"
   ],
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Grid=48x96; steps=700; time=2.1000\n",
      "Initial heavy-fluid area=1.000000000000\n",
      "Final heavy-fluid area=1.000000000000\n",
      "Relative mass error=4.441e-16\n",
      "Volume fraction range=[0.000000, 1.000000]\n",
      "Maximum projected divergence=2.675e-12\n",
      "Max speed=0.821857\n",
      "(wall-clock time of this run: 0.7 s)\n"
     ]
    }
   ],
   "execution_count": 8
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Results\n",
    "\n",
    "### Interface evolution\n",
    "Snapshots at $t\\approx0,\\;t_{end}/4,\\;t_{end}/2,\\;3t_{end}/4,\\;t_{end}$. Heavy fluid is blue, the black line is the $\\alpha=0.5$ contour."
   ]
  },
  {
   "cell_type": "code",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-27T17:49:00.104801Z",
     "iopub.status.busy": "2026-10-27T17:49:00.104801Z",
     "iopub.status.idle": "2026-10-27T17:49:00.718806Z",
     "shell.execute_reply": "2026-10-27T17:49:00.718806Z"
    },
    "ExecuteTime": {
     "end_time": "2026-10-27T17:49:59.765326Z",
     "start_time": "2026-10-27T17:49:59.150434Z"
    }
   },
   "source": [
    "fig, axs = plt.subplots(1, len(snapshots), figsize=(13, 4.8), sharex=True, sharey=True)\n",
    "for ax, (ti, ai) in zip(axs, snapshots):\n",
    "    ax.imshow(ai, origin=\"lower\", extent=(0, Lx, 0, Ly), cmap=\"Blues\", vmin=0, vmax=1)\n",
    "    ax.contour(x, y, ai, levels=[0.5], colors=\"k\", linewidths=0.8)\n",
    "    ax.set_title(f\"t = {ti:.2f} s\")\n",
    "    ax.set_xlabel(\"x (m)\")\n",
    "axs[0].set_ylabel(\"y (m)\")\n",
    "fig.suptitle(\"Experiment 4: Rayleigh-Taylor instability (VOF, heavy fluid in blue)\")\n",
    "fig.tight_layout()\n",
    "fig.savefig(\"experiment_4_rti_output.png\", dpi=180)\n",
    "plt.show()"
   ],
   "outputs": [
    {
     "data": {
      "text/plain": [
       "<Figure size 1300x480 with 5 Axes>"
      ],
      "image/png": "..."
     },
     "metadata": {},
     "output_type": "display_data"
    }
   ],
   "execution_count": 10
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": "### Numerical summary at the snapshot times"
  },
  {
   "cell_type": "code",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-27T17:49:00.718806Z",
     "iopub.status.busy": "2026-10-27T17:49:00.718806Z",
     "iopub.status.idle": "2026-10-27T17:49:00.732061Z",
     "shell.execute_reply": "2026-10-27T17:49:00.732061Z"
    },
    "ExecuteTime": {
     "end_time": "2026-10-27T17:49:59.788444Z",
     "start_time": "2026-10-27T17:49:59.774029Z"
    }
   },
   "source": [
    "print(f\"{'t (s)':>8}{'h_spike (m)':>13}{'h_bubble (m)':>14}{'amp a (m)':>11}{'max|u| (m/s)':>14}{'E_k (J/m)':>12}{'smear area':>12}\")\n",
    "for d in res[\"snap_diag\"]:\n",
    "    print(f\"{d['t']:>8.4f}{d['h_spike']:>13.4f}{d['h_bubble']:>14.4f}{d['amp']:>11.4f}{d['max_speed']:>14.5f}{d['ke']:>12.5f}{d['smear']:>12.5f}\")"
   ],
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "   t (s)  h_spike (m)  h_bubble (m)  amp a (m)  max|u| (m/s)   E_k (J/m)  smear area\n",
      "  0.0000       0.0407        0.0407     0.0407       0.00000     0.00000     0.02083\n",
      "  0.5250       0.0634        0.0634     0.0634       0.10504     0.00116     0.05903\n",
      "  1.0500       0.1502        0.1502     0.1502       0.26441     0.00938     0.12674\n",
      "  1.5750       0.3001        0.3001     0.3001       0.48319     0.04630     0.23611\n",
      "  2.1000       0.4769        0.4769     0.4769       0.82186     0.14321     0.41667\n"
     ]
    }
   ],
   "execution_count": 11
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### Time histories of the diagnostics\n",
    "Panels: (a) spike and bubble penetration, (b) maximum velocity, (c) kinetic energy (log scale), (d) heavy-fluid area error and maximum divergence after projection (log scale)."
   ]
  },
  {
   "cell_type": "code",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-27T17:49:00.732061Z",
     "iopub.status.busy": "2026-10-27T17:49:00.732061Z",
     "iopub.status.idle": "2026-10-27T17:49:01.892321Z",
     "shell.execute_reply": "2026-10-27T17:49:01.892321Z"
    },
    "ExecuteTime": {
     "end_time": "2026-10-27T17:50:01.026424Z",
     "start_time": "2026-10-27T17:49:59.794313Z"
    }
   },
   "source": [
    "t_h = hist[\"t\"]\n",
    "C1, C2, C3 = \"#0072B2\", \"#D55E00\", \"#009E73\"\n",
    "fig, axs = plt.subplots(2, 2, figsize=(11, 7.5))\n",
    "ax = axs[0, 0]\n",
    "ax.plot(t_h, hist[\"h_spike\"], color=C1, lw=2, label=\"spike (heavy, falling)\")\n",
    "ax.plot(t_h, hist[\"h_bubble\"], color=C2, lw=2, label=\"bubble (light, rising)\")\n",
    "ax.set_xlabel(\"t (s)\"); ax.set_ylabel(\"tip penetration (m)\"); ax.set_title(\"(a) Interface penetration\"); ax.legend(frameon=False)\n",
    "ax = axs[0, 1]\n",
    "ax.plot(t_h, hist[\"max_speed\"], color=C3, lw=2)\n",
    "ax.set_xlabel(\"t (s)\"); ax.set_ylabel(\"max |velocity| (m/s)\"); ax.set_title(\"(b) Maximum speed\")\n",
    "ax = axs[1, 0]\n",
    "ax.semilogy(t_h[1:], hist[\"ke\"][1:], color=C1, lw=2)\n",
    "ax.set_xlabel(\"t (s)\"); ax.set_ylabel(\"kinetic energy (J/m)\"); ax.set_title(\"(c) Kinetic energy\")\n",
    "ax = axs[1, 1]\n",
    "ax.semilogy(t_h[1:], np.abs(hist[\"mass\"][1:] - hist[\"mass\"][0]) / hist[\"mass\"][0] + 1e-18, color=C2, lw=1.5, label=\"relative mass error (+1e-18 floor)\")\n",
    "ax.semilogy(t_h[1:], hist[\"div\"][1:], color=C1, lw=1.5, label=\"max |div u| after projection\")\n",
    "ax.set_xlabel(\"t (s)\"); ax.set_title(\"(d) Conservation / constraint\"); ax.legend(frameon=False, fontsize=8)\n",
    "for a in axs.ravel():\n",
    "    a.grid(alpha=0.25)\n",
    "fig.tight_layout()\n",
    "fig.savefig(\"fig_diagnostics.png\", dpi=180)\n",
    "plt.show()\n",
    "print(f\"Peak max speed = {hist['max_speed'].max():.5f} m/s at t = {t_h[np.argmax(hist['max_speed'])]:.3f} s\")\n",
    "print(f\"Final E_k      = {hist['ke'][-1]:.5f} J/m\")\n",
    "print(f\"Largest relative mass deviation over run = {np.max(np.abs(hist['mass']-hist['mass'][0]))/hist['mass'][0]:.3e}\")\n",
    "print(f\"Largest post-projection |div u| over run = {hist['div'].max():.3e}\")"
   ],
   "outputs": [
    {
     "data": {
      "text/plain": [
       "<Figure size 1100x750 with 4 Axes>"
      ],
      "image/png": "..."
     },
     "metadata": {},
     "output_type": "display_data"
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Peak max speed = 0.82186 m/s at t = 2.100 s\n",
      "Final E_k      = 0.14321 J/m\n",
      "Largest relative mass deviation over run = 4.441e-16\n",
      "Largest post-projection |div u| over run = 2.675e-12\n"
     ]
    }
   ],
   "execution_count": 12
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### Early-time growth rate versus linear theory\n",
    "For motion starting from rest, linear theory predicts $a(t)=a_0\\cosh(\\sigma t)$. The growth rate $\\sigma$ is fitted by least squares to the measured amplitude $a(t)=\\tfrac12(h_s+h_b)$ over the window in which the amplitude is still small, defined as $a<a_{max}$. The window is varied ($a_{max}=0.08,\\,0.10,\\,0.12$ m, i.e. $0.08\\lambda$–$0.12\\lambda$) to show how sensitive the estimate is. The measured values are compared with the inviscid and approximate viscous theoretical rates computed in Section 2."
   ]
  },
  {
   "cell_type": "code",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-27T17:49:01.892321Z",
     "iopub.status.busy": "2026-10-27T17:49:01.892321Z",
     "iopub.status.idle": "2026-10-27T17:49:01.908036Z",
     "shell.execute_reply": "2026-10-27T17:49:01.908036Z"
    },
    "ExecuteTime": {
     "end_time": "2026-10-27T17:50:01.054163Z",
     "start_time": "2026-10-27T17:50:01.041431Z"
    }
   },
   "source": [
    "a0_meas = hist[\"amp\"][0]\n",
    "\n",
    "def fit_sigma(t, a, a_max, a0=None):\n",
    "    a0 = a[0] if a0 is None else a0\n",
    "    m = a <= a_max\n",
    "    tt, aa = t[m], a[m]\n",
    "    cost = lambda s: np.sum((aa - a0 * np.cosh(s * tt)) ** 2)\n",
    "    out = minimize_scalar(cost, bounds=(0.1, 6.0), method=\"bounded\")\n",
    "    return out.x, tt[-1], m.sum()\n",
    "\n",
    "print(f\"measured initial amplitude a0 = {a0_meas:.4f} m (nominal {amp0})\")\n",
    "print(f\"theory: sigma_inviscid = {sigma_inviscid:.4f} 1/s, sigma_viscous(approx) = {sigma_viscous:.4f} 1/s\\n\")\n",
    "fit_rows = []\n",
    "print(f\"{'a_max (m)':>10}{'t_window (s)':>14}{'n_points':>10}{'sigma_fit (1/s)':>17}{'/sigma_inv':>12}{'/sigma_visc':>13}\")\n",
    "for a_max in (0.08, 0.10, 0.12):\n",
    "    s, t_win, n = fit_sigma(t_h, hist[\"amp\"], a_max)\n",
    "    fit_rows.append(dict(a_max=a_max, t_window=float(t_win), n=int(n), sigma=float(s)))\n",
    "    print(f\"{a_max:>10.2f}{t_win:>14.3f}{n:>10d}{s:>17.4f}{s/sigma_inviscid:>12.3f}{s/sigma_viscous:>13.3f}\")\n",
    "sigma_fit = fit_rows[1][\"sigma\"]     # a_max = 0.10 (0.1 lambda) is the reference window"
   ],
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "measured initial amplitude a0 = 0.0407 m (nominal 0.04)\n",
      "theory: sigma_inviscid = 2.3672 1/s, sigma_viscous(approx) = 2.1780 1/s\n",
      "\n",
      " a_max (m)  t_window (s)  n_points  sigma_fit (1/s)  /sigma_inv  /sigma_visc\n",
      "      0.08         0.663       222           1.9177       0.810        0.881\n",
      "      0.10         0.798       267           1.9331       0.817        0.888\n",
      "      0.12         0.906       303           1.9298       0.815        0.886\n"
     ]
    }
   ],
   "execution_count": 13
  },
  {
   "cell_type": "code",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-27T17:49:01.908036Z",
     "iopub.status.busy": "2026-10-27T17:49:01.908036Z",
     "iopub.status.idle": "2026-10-27T17:49:02.304570Z",
     "shell.execute_reply": "2026-10-27T17:49:02.304570Z"
    },
    "ExecuteTime": {
     "end_time": "2026-10-27T17:50:01.487900Z",
     "start_time": "2026-10-27T17:50:01.072858Z"
    }
   },
   "source": [
    "fig, axs = plt.subplots(1, 2, figsize=(11, 4.3))\n",
    "tt = np.linspace(0, 1.2, 300)\n",
    "ax = axs[0]\n",
    "ax.plot(t_h, hist[\"amp\"], color=C1, lw=2, label=\"simulation  a(t)\")\n",
    "ax.plot(tt, a0_meas * np.cosh(sigma_inviscid * tt), \"--\", color=C2, lw=1.8, label=f\"a0 cosh(σ t), σ = {sigma_inviscid:.3f} (inviscid)\")\n",
    "ax.plot(tt, a0_meas * np.cosh(sigma_viscous * tt), \":\", color=C3, lw=2.2, label=f\"a0 cosh(σ t), σ = {sigma_viscous:.3f} (viscous approx.)\")\n",
    "ax.plot(tt, a0_meas * np.cosh(sigma_fit * tt), \"-.\", color=\"0.35\", lw=1.5, label=f\"fitted σ = {sigma_fit:.3f} (a ≤ 0.10 m)\")\n",
    "ax.axhline(0.1 * Lx, color=\"0.7\", lw=0.8)\n",
    "ax.text(0.02, 0.1 * Lx + 0.008, \"a = 0.1 λ\", color=\"0.4\", fontsize=8)\n",
    "ax.set_xlim(0, 1.2); ax.set_ylim(0, 0.45)\n",
    "ax.set_xlabel(\"t (s)\"); ax.set_ylabel(\"mean amplitude a (m)\"); ax.set_title(\"(a) Amplitude growth, linear scale\")\n",
    "ax.legend(frameon=False, fontsize=8, loc=\"upper left\"); ax.grid(alpha=0.25)\n",
    "ax = axs[1]\n",
    "ax.semilogy(t_h, hist[\"amp\"], color=C1, lw=2, label=\"simulation\")\n",
    "ax.semilogy(tt, a0_meas * np.cosh(sigma_inviscid * tt), \"--\", color=C2, lw=1.8, label=\"inviscid theory\")\n",
    "ax.semilogy(tt, a0_meas * np.cosh(sigma_viscous * tt), \":\", color=C3, lw=2.2, label=\"viscous approx.\")\n",
    "ax.set_xlim(0, t_end); ax.set_xlabel(\"t (s)\"); ax.set_ylabel(\"mean amplitude a (m)\"); ax.set_title(\"(b) Amplitude growth, log scale\")\n",
    "ax.legend(frameon=False, fontsize=8); ax.grid(alpha=0.25, which=\"both\")\n",
    "fig.tight_layout()\n",
    "fig.savefig(\"fig_growth.png\", dpi=180)\n",
    "plt.show()"
   ],
   "outputs": [
    {
     "data": {
      "text/plain": [
       "<Figure size 1100x430 with 2 Axes>"
      ],
      "image/png": "..."
     },
     "metadata": {},
     "output_type": "display_data"
    }
   ],
   "execution_count": 14
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### Final flow field\n",
    "Cell-centred velocity (interpolated from the staggered faces) at $t=t_{end}$, with the $\\alpha=0.5$ interface. Colour shows speed."
   ]
  },
  {
   "cell_type": "code",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-27T17:49:02.304570Z",
     "iopub.status.busy": "2026-10-27T17:49:02.304570Z",
     "iopub.status.idle": "2026-10-27T17:49:02.685431Z",
     "shell.execute_reply": "2026-10-27T17:49:02.685431Z"
    },
    "ExecuteTime": {
     "end_time": "2026-10-27T17:50:01.959089Z",
     "start_time": "2026-10-27T17:50:01.494493Z"
    }
   },
   "source": [
    "uc = 0.5 * (u + np.roll(u, -1, axis=1))\n",
    "vc = 0.5 * (v[1:] + v[:-1])\n",
    "speed = np.hypot(uc, vc)\n",
    "fig, axs = plt.subplots(1, 2, figsize=(10, 6.2), sharey=True)\n",
    "ax = axs[0]\n",
    "im = ax.imshow(speed, origin=\"lower\", extent=(0, Lx, 0, Ly), cmap=\"viridis\")\n",
    "ax.contour(x, y, alpha, levels=[0.5], colors=\"w\", linewidths=1.0)\n",
    "ax.set_title(\"(a) Speed |u| (m/s)  [white: α = 0.5]\"); ax.set_xlabel(\"x (m)\"); ax.set_ylabel(\"y (m)\")\n",
    "fig.colorbar(im, ax=ax, shrink=0.7)\n",
    "ax = axs[1]\n",
    "ax.imshow(alpha, origin=\"lower\", extent=(0, Lx, 0, Ly), cmap=\"Blues\", vmin=0, vmax=1)\n",
    "s = 2\n",
    "ax.quiver(x[::s], y[::s*1], uc[::s, ::s], vc[::s, ::s], color=\"k\", scale=12, width=0.003)\n",
    "ax.set_title(\"(b) Velocity vectors over α\"); ax.set_xlabel(\"x (m)\")\n",
    "fig.tight_layout()\n",
    "fig.savefig(\"fig_flow_field.png\", dpi=180)\n",
    "plt.show()\n",
    "iy, ix = np.unravel_index(np.argmax(speed), speed.shape)\n",
    "print(f\"Maximum cell-centred speed = {speed.max():.5f} m/s at (x, y) = ({x[ix]:.3f}, {y[iy]:.3f}) m\")\n",
    "sym = np.max(np.abs(alpha - alpha[:, ::-1]))\n",
    "print(f\"Left-right symmetry error of alpha, max|alpha(x) - alpha(Lx-x)| = {sym:.3e}\")\n",
    "print(f\"Interface smearing area (0.05<alpha<0.95): initial {hist['smear'][0]:.4f} m^2 -> final {hist['smear'][-1]:.4f} m^2\")"
   ],
   "outputs": [
    {
     "data": {
      "text/plain": [
       "<Figure size 1000x620 with 3 Axes>"
      ],
      "image/png": "..."
     },
     "metadata": {},
     "output_type": "display_data"
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Maximum cell-centred speed = 0.82171 m/s at (x, y) = (0.490, 1.010) m\n",
      "Left-right symmetry error of alpha, max|alpha(x) - alpha(Lx-x)| = 6.328e-15\n",
      "Interface smearing area (0.05<alpha<0.95): initial 0.0208 m^2 -> final 0.4167 m^2\n"
     ]
    }
   ],
   "execution_count": 15
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Grid-sensitivity study\n",
    "\n",
    "The same parameters are run on a coarser (24×48) and a finer (96×192) grid (the 48×96 result is reused). The time step follows the same adaptive formula, so it differs between grids when the viscous or CFL limits are tighter than 0.003 s. The comparison shows how much the first-order upwind VOF result depends on resolution."
   ]
  },
  {
   "cell_type": "code",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-27T17:49:02.686438Z",
     "iopub.status.busy": "2026-10-27T17:49:02.686438Z",
     "iopub.status.idle": "2026-10-27T17:49:05.778169Z",
     "shell.execute_reply": "2026-10-27T17:49:05.778169Z"
    },
    "ExecuteTime": {
     "end_time": "2026-10-27T17:50:05.415541Z",
     "start_time": "2026-10-27T17:50:01.968088Z"
    }
   },
   "source": [
    "runs = {\"24x48\": run_rti(24, 48, Lx, Ly, t_end, nu, g, rho_light, rho_heavy, amp0),\n",
    "        \"48x96\": res,\n",
    "        \"96x192\": run_rti(96, 192, Lx, Ly, t_end, nu, g, rho_light, rho_heavy, amp0)}\n",
    "grid_rows = []\n",
    "print(f\"{'grid':>8}{'steps':>7}{'h_spike':>9}{'h_bubble':>10}{'max|u|':>9}{'E_k':>9}{'smear':>9}{'sigma_fit':>10}{'mass err':>11}{'max div':>11}{'wall (s)':>9}\")\n",
    "for name, r in runs.items():\n",
    "    h = r[\"hist\"]\n",
    "    s_fit, _, _ = fit_sigma(h[\"t\"], h[\"amp\"], 0.10)\n",
    "    row = dict(grid=name, steps=r[\"steps\"], h_spike=float(h[\"h_spike\"][-1]), h_bubble=float(h[\"h_bubble\"][-1]),\n",
    "               max_speed=float(h[\"max_speed\"][-1]), ke=float(h[\"ke\"][-1]), smear=float(h[\"smear\"][-1]),\n",
    "               sigma_fit=float(s_fit),\n",
    "               mass_err=float(abs(r[\"mass\"] - r[\"initial_mass\"]) / r[\"initial_mass\"]),\n",
    "               max_div=float(r[\"maximum_divergence\"]), wall=float(r[\"wall\"]))\n",
    "    grid_rows.append(row)\n",
    "    print(f\"{name:>8}{row['steps']:>7d}{row['h_spike']:>9.4f}{row['h_bubble']:>10.4f}{row['max_speed']:>9.4f}{row['ke']:>9.4f}{row['smear']:>9.4f}{row['sigma_fit']:>10.4f}{row['mass_err']:>11.2e}{row['max_div']:>11.2e}{row['wall']:>9.1f}\")\n",
    "print(f\"\\ntheory: sigma_inviscid = {sigma_inviscid:.4f}, sigma_viscous(approx) = {sigma_viscous:.4f}\")"
   ],
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "    grid  steps  h_spike  h_bubble   max|u|      E_k    smear sigma_fit   mass err    max div wall (s)\n",
      "   24x48    700   0.4664    0.4664   0.7432   0.1188   0.5278    1.8333   3.33e-16   1.23e-13      0.3\n",
      "   48x96    700   0.4769    0.4769   0.8219   0.1432   0.4167    1.9331   4.44e-16   2.68e-12      0.7\n",
      "  96x192    861   0.4809    0.4809   0.8750   0.1586   0.3082    1.9859   1.67e-15   2.70e-11      3.0\n",
      "\n",
      "theory: sigma_inviscid = 2.3672, sigma_viscous(approx) = 2.1780\n"
     ]
    }
   ],
   "execution_count": 16
  },
  {
   "cell_type": "code",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-27T17:49:05.778169Z",
     "iopub.status.busy": "2026-10-27T17:49:05.778169Z",
     "iopub.status.idle": "2026-10-27T17:49:06.175116Z",
     "shell.execute_reply": "2026-10-27T17:49:06.175116Z"
    },
    "ExecuteTime": {
     "end_time": "2026-10-27T17:50:05.880980Z",
     "start_time": "2026-10-27T17:50:05.435641Z"
    }
   },
   "source": [
    "fig, axs = plt.subplots(1, 3, figsize=(14, 4.6))\n",
    "cols = {\"24x48\": \"#009E73\", \"48x96\": \"#0072B2\", \"96x192\": \"#D55E00\"}\n",
    "ax = axs[0]\n",
    "for name, r in runs.items():\n",
    "    ax.plot(r[\"hist\"][\"t\"], r[\"hist\"][\"h_spike\"], color=cols[name], lw=2, label=name)\n",
    "ax.set_title(\"(a) Spike penetration\"); ax.set_xlabel(\"t (s)\"); ax.set_ylabel(\"h_spike (m)\"); ax.legend(frameon=False); ax.grid(alpha=0.25)\n",
    "ax = axs[1]\n",
    "for name, r in runs.items():\n",
    "    ax.plot(r[\"hist\"][\"t\"], r[\"hist\"][\"h_bubble\"], color=cols[name], lw=2, label=name)\n",
    "ax.set_title(\"(b) Bubble penetration\"); ax.set_xlabel(\"t (s)\"); ax.set_ylabel(\"h_bubble (m)\"); ax.legend(frameon=False); ax.grid(alpha=0.25)\n",
    "ax = axs[2]\n",
    "for name, r in runs.items():\n",
    "    ax.contour(r[\"x\"], r[\"y\"], r[\"alpha\"], levels=[0.5], colors=[cols[name]], linewidths=1.6)\n",
    "ax.plot([], [], color=cols[\"24x48\"], label=\"24x48\"); ax.plot([], [], color=cols[\"48x96\"], label=\"48x96\"); ax.plot([], [], color=cols[\"96x192\"], label=\"96x192\")\n",
    "ax.set_xlim(0, Lx); ax.set_ylim(0.2, 1.8); ax.set_aspect(\"equal\")\n",
    "ax.set_title(f\"(c) Interface (α = 0.5) at t = {t_end} s\"); ax.set_xlabel(\"x (m)\"); ax.set_ylabel(\"y (m)\"); ax.legend(frameon=False, loc=\"lower left\")\n",
    "fig.tight_layout()\n",
    "fig.savefig(\"fig_grid_study.png\", dpi=180)\n",
    "plt.show()"
   ],
   "outputs": [
    {
     "data": {
      "text/plain": [
       "<Figure size 1400x460 with 3 Axes>"
      ],
      "image/png": "..."
     },
     "metadata": {},
     "output_type": "display_data"
    }
   ],
   "execution_count": 17
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.10.11"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}


{'cells': [{'cell_type': 'markdown',
   'metadata': {},
   'source': ['07_Shallow_Water_Wave# Experiment 8 — Rayleigh–Taylor Instability with a Volume-of-Fluid Finite-Volume Solver\n',
    '\n',
    '\n',
    'It simulates the **Rayleigh–Taylor instability (RTI)** of a heavy fluid resting on a light fluid in a\n',
    '2-D box, using\n',
    '\n',
    '* a **volume-of-fluid (VOF)** colour function advected with a first-order **upwind** finite-volume scheme,\n',
    '* a **staggered (MAC) grid** incompressible Navier–Stokes solver,\n',
    '* a **Chorin projection** with a sparse-direct Poisson solve,\n',
    '* the **Boussinesq approximation** for the density difference.\n']},
  {'cell_type': 'code',
   'metadata': {'execution': {'iopub.execute_input': '2026-10-27T17:48:58.746913Z',
     'iopub.status.busy': '2026-10-27T17:48:58.746913Z',
     'iopub.status.idle': '2026-10-27T17:48:59.281221Z',
     'shell.execute_reply': '2026-10-27T17:48:59.281221Z'},
    'ExecuteTime': {'end_time': 

## Parameters

Use `quick` to test the workflow, `lab` for the prescribed case, and `custom` only when changing the form values for the assignment.

## Run the solver

## Numerical summary

## Quantitative plots

## Inline field visualization

This preview removes the need for ParaView during the basic laboratory workflow. The VTK file is still included in the downloaded results for optional advanced post-processing.

## Download results

## 7. Compulsory assignment

Run $Re=100$ and $Re=400$. Compare centreline velocities, main-vortex position, convergence history, and secondary-corner-vortex development.

## Report checklist

- State the governing equations and boundary conditions.
- Report the complete parameter set and grid.
- Include residual or error history.
- Include the required quantitative comparison.
- Include at least one correctly labelled inline field figure; VTK/ParaView figures are optional unless specifically assigned.
- Distinguish observed numerical behaviour from physical interpretation.
- State the limitations of the supplied teaching model.